# **Problem - 10 : Highest Salary in Each Department.**

In [0]:
# Define the schema for the employees DataFrame
schema = ["emp_id", "emp_name", "department", "salary"]

# Sample employee data with emp_id, emp_name, department, and salary
data = [
    (101, "John",   "IT",       90000),
    (102, "Alice",  "IT",       85000),
    (103, "Bob",    "IT",       90000),
    (104, "David",  "HR",       70000),
    (105, "Emma",   "HR",       75000),
    (106, "Sophia", "Finance",  95000),
    (107, "James",  "Finance",  88000),
    (108, "Olivia", "Sales",    65000),
    (109, "Lucas",  "Sales",    65000),
    (110, "Mason",  "Marketing",72000)
]

# Create a Spark DataFrame using the schema and data
employees = spark.createDataFrame(schema = schema, data = data)

# Display the employees DataFrame
display(employees)

emp_id,emp_name,department,salary
101,John,IT,90000
102,Alice,IT,85000
103,Bob,IT,90000
104,David,HR,70000
105,Emma,HR,75000
106,Sophia,Finance,95000
107,James,Finance,88000
108,Olivia,Sales,65000
109,Lucas,Sales,65000
110,Mason,Marketing,72000


In [0]:
# Find employees with the highest salary in each department using dense rank
from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank
from pyspark.sql.functions import col

# Define window specification to rank salaries within each department
window_spec = Window.partitionBy("department") \
                    .orderBy(col("salary").desc())

# Add rank column, filter for top salary (rank 1), and drop the rank column
result = employees.withColumn(
    "rank",
    dense_rank().over(window_spec)
).filter(
    col("rank") == 1
).drop("rank")

# Display the result DataFrame
display(result)

emp_id,emp_name,department,salary
106,Sophia,Finance,95000
105,Emma,HR,75000
101,John,IT,90000
103,Bob,IT,90000
110,Mason,Marketing,72000
108,Olivia,Sales,65000
109,Lucas,Sales,65000


In [0]:
# Find employees with the highest salary in each department using groupBy and join
from pyspark.sql.functions import max
from pyspark.sql.functions import col

# Compute the maximum salary for each department
max_salary = employees.groupBy("department") \
    .agg(max("salary").alias("max_salary"))

# Create aliases for employees and max_salary DataFrames
emp_alias = employees.alias("emp")
max_alias = max_salary.alias("max_sal")

# Join employees with max_salary to get employees having the highest salary in their department
result = emp_alias.join(
    max_alias,
    (col("emp.department") == col("max_sal.department")) &
    (col("emp.salary") == col("max_sal.max_salary")),
    "inner"
).select(
    col("emp.emp_id"),
    col("emp.emp_name"),
    col("emp.department"),
    col("emp.salary")
)

# Display the result DataFrame
display(result)

emp_id,emp_name,department,salary
101,John,IT,90000
103,Bob,IT,90000
105,Emma,HR,75000
106,Sophia,Finance,95000
108,Olivia,Sales,65000
109,Lucas,Sales,65000
110,Mason,Marketing,72000


In [0]:
# Register the employees DataFrame as a temporary SQL view for querying
employees.createOrReplaceTempView("employees")

In [0]:
%sql
-- Find employees with the highest salary in each department using DENSE_RANK
WITH cte AS
(
    SELECT *,
           DENSE_RANK() OVER (
               PARTITION BY department
               ORDER BY salary DESC
           ) AS rnk
    FROM employees
)
SELECT emp_id,
       emp_name,
       department,
       salary
FROM cte
WHERE rnk = 1;

emp_id,emp_name,department,salary
106,Sophia,Finance,95000
105,Emma,HR,75000
101,John,IT,90000
103,Bob,IT,90000
110,Mason,Marketing,72000
108,Olivia,Sales,65000
109,Lucas,Sales,65000


In [0]:
%sql
-- Find employees with the highest salary in each department using DENSE_RANK
SELECT emp_id,
       emp_name,
       department,
       salary
FROM
(
    SELECT *,
           DENSE_RANK() OVER (
               PARTITION BY department
               ORDER BY salary DESC
           ) AS rnk
    FROM employees
) t
WHERE rnk = 1;

emp_id,emp_name,department,salary
106,Sophia,Finance,95000
105,Emma,HR,75000
101,John,IT,90000
103,Bob,IT,90000
110,Mason,Marketing,72000
108,Olivia,Sales,65000
109,Lucas,Sales,65000
